# Lesson 12D: Generative Adversarial Networks - Practical Implementation

## Introduction

**Reference**: Goodfellow, I., Pouget-Abadie, J., Mirza, M., Xu, B., Warde-Farley, D., Ozair, S., ... & Bengio, Y. (2014). "Generative Adversarial Nets". *Advances in Neural Information Processing Systems (NeurIPS)*, 27, 2672-2680.

Generative Adversarial Networks (GANs) learn to generate data through an adversarial process between two neural networks: a generator G that creates synthetic samples, and a discriminator D that attempts to distinguish real from fake samples.

**Key Innovation**: Training via a minimax game rather than explicit likelihood maximization.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons

torch.manual_seed(42)
np.random.seed(42)
print("Libraries loaded successfully")

## Theoretical Foundation

### Minimax Objective

The GAN objective is a two-player minimax game:

$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{\text{data}}(x)}[\log D(x)] + \mathbb{E}_{z \sim p_z(z)}[\log(1 - D(G(z)))]$$

where:
- D(x): Discriminator's estimate that x is real
- G(z): Generator's output given noise z
- p_data(x): True data distribution
- p_z(z): Prior noise distribution (typically N(0, I))

### Optimal Discriminator

For fixed G, the optimal discriminator is:

$$D^*(x) = \frac{p_{\text{data}}(x)}{p_{\text{data}}(x) + p_g(x)}$$

where p_g is the distribution induced by the generator.

### Global Optimum

At the global optimum where p_g = p_data:
- D*(x) = 1/2 everywhere
- The minimax objective equals -log(4)

### Training Algorithm

**For each training iteration**:

1. **Update Discriminator**:
   - Sample minibatch of m noise samples {z⁽¹⁾, ..., z⁽ᵐ⁾} from p_z
   - Sample minibatch of m real samples {x⁽¹⁾, ..., x⁽ᵐ⁾} from p_data
   - Update D by ascending stochastic gradient:
   
   $$\nabla_{\theta_d} \frac{1}{m} \sum_{i=1}^m [\log D(x^{(i)}) + \log(1 - D(G(z^{(i)})))]$$

2. **Update Generator**:
   - Sample minibatch of m noise samples from p_z
   - Update G by descending stochastic gradient:
   
   $$\nabla_{\theta_g} \frac{1}{m} \sum_{i=1}^m \log(1 - D(G(z^{(i)})))$$
   
   **Note**: In practice, we maximize log D(G(z)) instead to avoid saturating gradients.

In [ ]:
class Generator(nn.Module):
    """
    Generator network G(z): Z → X
    Maps noise z to data space
    """
    def __init__(self, latent_dim=10, hidden_dim=64, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, z):
        """
        Args:
            z: Noise vector [batch_size, latent_dim]
        Returns:
            x_fake: Generated sample [batch_size, output_dim]
        """
        return self.net(z)

class Discriminator(nn.Module):
    """
    Discriminator network D(x): X → [0,1]
    Estimates probability that x is real
    """
    def __init__(self, input_dim=2, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        """
        Args:
            x: Input sample [batch_size, input_dim]
        Returns:
            prob: Probability x is real [batch_size, 1]
        """
        return self.net(x)

# Initialize networks
latent_dim = 10
G = Generator(latent_dim=latent_dim, hidden_dim=64, output_dim=2)
D = Discriminator(input_dim=2, hidden_dim=64)

print("Generator architecture:")
print(f"  Parameters: {sum(p.numel() for p in G.parameters()):,}")
print(f"\nDiscriminator architecture:")
print(f"  Parameters: {sum(p.numel() for p in D.parameters()):,}")

## Dataset Preparation

We use the two-moons dataset as our target distribution p_data.

In [ ]:
# Generate target distribution
n_samples = 2000
X_real, _ = make_moons(n_samples=n_samples, noise=0.05, random_state=42)

print(f"Real data shape: {X_real.shape}")
print(f"Data range: [{X_real.min():.2f}, {X_real.max():.2f}]")

# Visualize target distribution
plt.figure(figsize=(8, 6))
plt.scatter(X_real[:, 0], X_real[:, 1], s=20, alpha=0.6, c='blue', label='Real data (p_data)')
plt.xlabel('x₁', fontsize=12)
plt.ylabel('x₂', fontsize=12)
plt.title('Target Distribution: Two Moons', fontweight='bold', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Convert to tensor
X_real = torch.FloatTensor(X_real)
print("Dataset prepared")

## Loss Functions and Optimization

We use Binary Cross-Entropy (BCE) as the loss function, which corresponds to the minimax objective.

In [ ]:
# Loss function
criterion = nn.BCELoss()

# Optimizers (using Adam as per DCGAN recommendations)
lr = 2e-4
beta1 = 0.5  # Momentum term
opt_G = optim.Adam(G.parameters(), lr=lr, betas=(beta1, 0.999))
opt_D = optim.Adam(D.parameters(), lr=lr, betas=(beta1, 0.999))

print(f"Optimization configuration:")
print(f"  Learning rate: {lr}")
print(f"  Optimizer: Adam (β₁={beta1}, β₂=0.999)")
print(f"  Loss: Binary Cross-Entropy")

## Training Procedure

We alternate between k=1 discriminator updates and 1 generator update.

In [ ]:
# Training configuration
batch_size = 64
n_epochs = 1000
k_disc = 1  # Number of discriminator updates per generator update

# Loss tracking
d_losses = []
g_losses = []
d_real_acc = []  # D(x) where x is real
d_fake_acc = []  # D(G(z))

print(f"Training GAN for {n_epochs} epochs...")
print(f"Batch size: {batch_size}")
print(f"Discriminator updates per G update: {k_disc}\n")

for epoch in range(n_epochs):
    # === Train Discriminator ===
    for _ in range(k_disc):
        # Sample real data
        idx = torch.randint(0, len(X_real), (batch_size,))
        real_samples = X_real[idx]
        real_labels = torch.ones(batch_size, 1)
        
        # Sample fake data
        z = torch.randn(batch_size, latent_dim)
        fake_samples = G(z)
        fake_labels = torch.zeros(batch_size, 1)
        
        # Discriminator forward pass
        D_real = D(real_samples)
        D_fake = D(fake_samples.detach())  # Detach to avoid backprop through G
        
        # Discriminator loss: maximize log(D(x)) + log(1-D(G(z)))
        loss_D_real = criterion(D_real, real_labels)
        loss_D_fake = criterion(D_fake, fake_labels)
        loss_D = loss_D_real + loss_D_fake
        
        # Backpropagation
        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()
    
    # === Train Generator ===
    z = torch.randn(batch_size, latent_dim)
    fake_samples = G(z)
    D_fake = D(fake_samples)
    
    # Generator loss: maximize log(D(G(z))) ≡ minimize -log(D(G(z)))
    # In practice: minimize log(1 - D(G(z))) or maximize log(D(G(z)))
    loss_G = criterion(D_fake, real_labels)  # Trick: use real_labels for G
    
    # Backpropagation
    opt_G.zero_grad()
    loss_G.backward()
    opt_G.step()
    
    # Logging
    d_losses.append(loss_D.item())
    g_losses.append(loss_G.item())
    d_real_acc.append(D_real.mean().item())
    d_fake_acc.append(D_fake.mean().item())
    
    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}")
        print(f"  D Loss: {loss_D.item():.4f}, G Loss: {loss_G.item():.4f}")
        print(f"  D(x_real): {D_real.mean().item():.4f}, D(G(z)): {D_fake.mean().item():.4f}")

print("\nTraining completed")
print(f"Final D(x_real): {d_real_acc[-1]:.4f} (should be ≈ 0.5 at equilibrium)")
print(f"Final D(G(z)): {d_fake_acc[-1]:.4f} (should be ≈ 0.5 at equilibrium)")

## Training Dynamics Analysis

In [ ]:
# Visualize training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax = axes[0]
ax.plot(d_losses, label='Discriminator Loss', linewidth=2, alpha=0.7)
ax.plot(g_losses, label='Generator Loss', linewidth=2, alpha=0.7)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('GAN Training Losses', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

# Discriminator outputs
ax = axes[1]
ax.plot(d_real_acc, label='D(x_real)', linewidth=2, alpha=0.7)
ax.plot(d_fake_acc, label='D(G(z))', linewidth=2, alpha=0.7)
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=2, 
          label='Equilibrium (0.5)')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Discriminator Output', fontsize=12)
ax.set_title('Discriminator Convergence', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()
print("Training dynamics visualized")

## Generation Quality Evaluation

In [ ]:
# Generate samples from trained generator
G.eval()
with torch.no_grad():
    z = torch.randn(2000, latent_dim)
    fake_samples = G(z).numpy()

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Real data
ax = axes[0]
ax.scatter(X_real[:, 0], X_real[:, 1], s=20, alpha=0.6, c='blue')
ax.set_xlabel('x₁', fontsize=11)
ax.set_ylabel('x₂', fontsize=11)
ax.set_title('Real Data (p_data)', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

# Generated data
ax = axes[1]
ax.scatter(fake_samples[:, 0], fake_samples[:, 1], s=20, alpha=0.6, c='red')
ax.set_xlabel('x₁', fontsize=11)
ax.set_ylabel('x₂', fontsize=11)
ax.set_title('Generated Data (p_g)', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

# Overlay comparison
ax = axes[2]
ax.scatter(X_real[:1000, 0], X_real[:1000, 1], s=20, alpha=0.4, c='blue', label='Real')
ax.scatter(fake_samples[:1000, 0], fake_samples[:1000, 1], s=20, alpha=0.4, c='red', label='Generated')
ax.set_xlabel('x₁', fontsize=11)
ax.set_ylabel('x₂', fontsize=11)
ax.set_title('Distribution Comparison', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("GAN successfully approximated target distribution")

## Discussion

### Training Insights

1. **Nash Equilibrium**: Ideally, D(x) ≈ 0.5 for both real and fake samples
2. **Training Instability**: GANs are sensitive to hyperparameters and can exhibit:
   - Mode collapse: G produces limited variety
   - Non-convergence: Losses oscillate indefinitely
   - Vanishing gradients: When D is too strong

### Theoretical Analysis

**Jensen-Shannon Divergence**: At optimality, the GAN objective minimizes:

$$C(G) = \max_D V(D, G) = 2 \cdot D_{JS}(p_{\text{data}} \| p_g) - 2\log 2$$

where D_JS is the Jensen-Shannon divergence.

**Training Challenges**:
1. **Vanishing Gradients**: When D is perfect, ∇_G log(1 - D(G(z))) → 0
2. **Mode Collapse**: G maps multiple z to same x (lacks diversity)
3. **Instability**: No guaranteed convergence (non-convex minimax game)

### Practical Improvements

1. **DCGAN** (Radford et al., 2015):
   - Use strided convolutions instead of pooling
   - Batch normalization in both G and D
   - LeakyReLU in D, ReLU in G
   - Adam optimizer with β₁ = 0.5

2. **Wasserstein GAN** (Arjovsky et al., 2017):
   - Replace JS divergence with Wasserstein distance
   - Improves training stability
   - Meaningful loss metric

3. **Progressive GAN** (Karras et al., 2018):
   - Grow networks progressively
   - Generates high-resolution images

4. **StyleGAN** (Karras et al., 2019):
   - Style-based generator architecture
   - State-of-the-art image synthesis

## References

1. Goodfellow, I., Pouget-Abadie, J., Mirza, M., et al. (2014). "Generative Adversarial Nets". *Advances in Neural Information Processing Systems (NeurIPS)*, 27, 2672-2680.

2. Radford, A., Metz, L., & Chintala, S. (2015). "Unsupervised Representation Learning with Deep Convolutional Generative Adversarial Networks". *arXiv preprint arXiv:1511.06434*.

3. Arjovsky, M., Chintala, S., & Bottou, L. (2017). "Wasserstein GAN". *International Conference on Machine Learning (ICML)*, 214-223.

4. Salimans, T., Goodfellow, I., Zaremba, W., et al. (2016). "Improved Techniques for Training GANs". *Advances in Neural Information Processing Systems (NeurIPS)*, 29, 2234-2242.

5. Karras, T., Laine, S., & Aila, T. (2019). "A Style-Based Generator Architecture for Generative Adversarial Networks". *IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)*, 4401-4410.

6. Goodfellow, I. (2016). "NIPS 2016 Tutorial: Generative Adversarial Networks". *arXiv preprint arXiv:1701.00160*.